In [ ]:
import os, uuid, hashlib, platform
# ===== 1. 填写学号 =====
STUDENT_ID = "20244081128"
NAME = "唐仕江"
# ===== 2. 打印 conda 环境 =====
# 注意：CONDA_DEFAULT_ENV 是 conda activate 时注入的变量，
# Jupyter 内核不经过 activate，因此为空时回退为 "base"
CONDA_ENV = os.environ.get("CONDA_DEFAULT_ENV") or "base"
print(f"[Conda 环境] {CONDA_ENV}")
# ===== 3. 机器指纹（绑定当前电脑）=====
MACHINE_FP = hashlib.sha256(
    f"{platform.node()}|{platform.machine()}|{platform.system()}|{uuid.getnode()}".encode()
).hexdigest()[:16]
# ===== 4. 生成唯一 ID =====
RUN_ID = f"{STUDENT_ID}-{MACHINE_FP}"
print(f"[学号]     {STUDENT_ID}")
print(f"[姓名]     {NAME}")
print(f"[机器指纹] {MACHINE_FP}")
print(f"[RUN_ID]   {RUN_ID}")

In [ ]:
#围绕半圆定律
import numpy as np
import matplotlib.pyplot as plt

#任务1
def wigner_density(x):
    #半圆密度，|x|大于2时是0
    if np.abs(x) <= 2:
        return np.sqrt(4 - x ** 2) / (2 * np.pi)
    return 0.0

print(wigner_density(0), wigner_density(1), wigner_density(3))
print()

#任务2
def generate_wigner(n):
    #上三角元素独立抽N(0,1/n)，再对称过来
    if isinstance(n, bool) or not isinstance(n, int) or n <= 0:
        raise ValueError("n必须是正整数")
    z = np.random.normal(0, 1 / np.sqrt(n), (n, n))
    return np.triu(z) + np.triu(z, 1).T

print(generate_wigner(3))
try:
    generate_wigner(-1)
except ValueError as e:
    print("捕获到异常：", e)
print()

#任务3
def get_spectrum(matrix):
    #返回全部特征值，eigvalsh本身按从小到大排
    return np.linalg.eigvalsh(matrix)

print(get_spectrum(generate_wigner(3)))
print()

#任务4
#四张子图分别画n=100、200、500、1000的标准化直方图，叠红色半圆曲线
xs = np.linspace(-2, 2, 200)
plt.figure(figsize=(6, 10))
for i, n in enumerate([100, 200, 500, 1000], start=1):
    plt.subplot(4, 1, i)
    eig = get_spectrum(generate_wigner(n))
    plt.hist(eig, bins=60, density=True, color="blue")
    plt.plot(xs, [wigner_density(v) for v in xs], "r")
    plt.title(f"n={n}")
plt.tight_layout()
plt.show()
#n=100时直方图就已经和半圆曲线基本贴合，n越大贴合得越好

In [ ]:
#Conway的生命游戏
import numpy as np
import matplotlib.pyplot as plt

#任务1
def is_valid_board(board):
    #二维且元素只有0和1才算有效棋盘
    if not isinstance(board, np.ndarray):
        return False
    if board.ndim != 2:
        return False
    return bool(np.isin(board, [0, 1]).all())

print(is_valid_board(np.array([[0, 1], [1, 0]])))
print(is_valid_board(np.array([[0, 2], [1, 0]])))
print(is_valid_board(np.array([1, 0, 1])))
print()

#任务2
def gol_step(board):
    #先检查棋盘，无效就抛异常
    if not is_valid_board(board):
        raise ValueError("不是有效的生命游戏棋盘")
    #朝8个方向平移后相加就是每个细胞的活邻居数，np.roll自带环绕边界
    neighbor = sum(np.roll(np.roll(board, i, 0), j, 1)
                   for i in (-1, 0, 1) for j in (-1, 0, 1) if (i, j) != (0, 0))
    #死细胞刚好3个活邻居复活，活细胞有2或3个活邻居存活
    return ((neighbor == 3) | ((board == 1) & (neighbor == 2))).astype(int)

#用一个滑翔机演示一步
board = np.zeros((8, 8), dtype=int)
board[1:4, 1:4] = [[0, 1, 0], [0, 0, 1], [1, 1, 1]]
print(board)
print(gol_step(board))
print("活细胞数：", board.sum(), "->", gol_step(board).sum())
try:
    gol_step(np.array([[0, 2], [1, 0]]))
except ValueError as e:
    print("捕获到异常：", e)
print()

#任务3
def draw_gol_board(board):
    #活细胞黑色死细胞白色，用反色gray_r
    plt.imshow(board, cmap="gray_r")
    plt.xticks([])
    plt.yticks([])

plt.figure(figsize=(8, 4))
plt.subplot(1, 2, 1)
draw_gol_board(board)
plt.subplot(1, 2, 2)
draw_gol_board(gol_step(board))
plt.show()

In [ ]:
#使用pandas DataFrame
import pandas as pd
import requests

#任务1
#老师说不能用拖文件的方式，要用requests.get()下载数据
url = "把老师给的数据直链填到这里"
try:
    r = requests.get(url, timeout=10)
    #状态码不是200就抛异常，防止把错误页面存成csv
    r.raise_for_status()
    with open("Iris.csv", "wb") as f:
        f.write(r.content)
    print("用get()下载成功，状态码：", r.status_code)
except Exception as e:
    print("get()下载失败，用本地已有的Iris.csv：", e)

df = pd.read_csv("Iris.csv")

df = pd.read_csv("Iris.csv")
print(df.shape)
print(df.dtypes)
print(df.columns.tolist())
print(df["Species"].nunique())
print(df["Species"].unique())
#150个数据点，3种花
print()

#任务2
#第35行（索引34）第四项测量有误，第38行（索引37）第二、三项有误
df.loc[34, ["SepalLengthCm", "SepalWidthCm", "PetalLengthCm", "PetalWidthCm"]] = [4.9, 3.1, 1.5, 0.2]
df.loc[37, ["SepalLengthCm", "SepalWidthCm", "PetalLengthCm", "PetalWidthCm"]] = [4.9, 3.6, 1.4, 0.1]
print(df.loc[34])
print(df.loc[37])
print()

#任务3
df["Petal.Ratio"] = df["PetalLengthCm"] / df["PetalWidthCm"]
df["Sepal.Ratio"] = df["SepalLengthCm"] / df["SepalWidthCm"]
print(df[["Petal.Ratio", "Sepal.Ratio"]].head())
print()

#任务4
df.to_csv("iris_corrected.csv", index=False)
print("已保存iris_corrected.csv")
print()

#任务5
stat = df.groupby("Species")[["Petal.Ratio", "Sepal.Ratio"]].agg(["mean", "median", "min", "max", "std"])
print(stat.round(2))